# 01. V-World Aerial Image Download

Download V-World satellite/aerial tiles for a target area and merge into a single GeoTIFF.

**Requirements**
- V-World API key: set `VWORLD_API_KEY` in `.env` file at project root
- `python-dotenv`: loads the API key from `.env`
- Conda environment: `svi_segformer`

**Output**
- `data/raw/vworld_aerial_{area_name}.tif` — merged GeoTIFF (EPSG:4326)
- `data/raw/vworld_aerial_{area_name}_metadata.json` — source, tile, and georeference metadata

In [ ]:
import json
import math
import os
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
import requests
import numpy as np
from PIL import Image
from io import BytesIO
import rasterio
from rasterio.transform import from_bounds
from rasterio.crs import CRS
from pathlib import Path

# API 키는 .env 파일에서 읽기
# 프로젝트 루트에 .env 파일 생성: VWORLD_API_KEY=your_key_here
from dotenv import load_dotenv
load_dotenv(Path("../../.env"))

API_KEY = os.environ.get("VWORLD_API_KEY")
if not API_KEY or API_KEY == "YOUR_VWORLD_API_KEY":
    raise RuntimeError("Set a valid VWORLD_API_KEY in ../../.env before running this notebook.")

# ── Config ──────────────────────────────────────────────────────────────────
ZOOM = 19           # 19 ≈ 0.25m/px (최고해상도)
AREA_NAME = "seodaemun_sinchon_urban"
AREA_DESCRIPTION = "Seodaemun-gu Sinchon / Ewha / south Yeonhui urban pilot area, excluding Ansan mountain"

# 관심 지역 경계 (WGS84 위경도) — 서대문구 신촌/이대/연희동 남측 도시부, 안산 산림 제외
BBOX = {
    "min_lon": 126.930,
    "min_lat":  37.555,
    "max_lon": 126.945,
    "max_lat":  37.565,
}

OUTPUT_DIR = Path("../../data/raw")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / f"vworld_aerial_{AREA_NAME}.tif"
METADATA_PATH = OUTPUT_DIR / f"vworld_aerial_{AREA_NAME}_metadata.json"

TILE_URL = "https://api.vworld.kr/req/wmts/1.0.0/{key}/Satellite/{z}/{y}/{x}.jpeg"
CAPABILITIES_URL = "https://api.vworld.kr/req/wmts/1.0.0/{key}/WMTSCapabilities.xml"
TILE_SIZE = 256

In [ ]:
def lon_to_tile_x(lon: float, zoom: int) -> int:
    return int((lon + 180) / 360 * 2**zoom)

def lat_to_tile_y(lat: float, zoom: int) -> int:
    lat_r = math.radians(lat)
    return int((1 - math.log(math.tan(lat_r) + 1 / math.cos(lat_r)) / math.pi) / 2 * 2**zoom)

def tile_to_lon(x: int, zoom: int) -> float:
    return x / 2**zoom * 360 - 180

def tile_to_lat(y: int, zoom: int) -> float:
    n = math.pi - 2 * math.pi * y / 2**zoom
    return math.degrees(math.atan(math.sinh(n)))

x_min = lon_to_tile_x(BBOX["min_lon"], ZOOM)
x_max = lon_to_tile_x(BBOX["max_lon"], ZOOM)
y_min = lat_to_tile_y(BBOX["max_lat"], ZOOM)  # lat↑ → tile_y↓
y_max = lat_to_tile_y(BBOX["min_lat"], ZOOM)

n_x = x_max - x_min + 1
n_y = y_max - y_min + 1
print(f"Tile grid: {n_x} x {n_y} = {n_x * n_y} tiles")
print(f"Canvas size: {n_x * TILE_SIZE} x {n_y * TILE_SIZE} px")

In [ ]:
canvas = np.zeros((n_y * TILE_SIZE, n_x * TILE_SIZE, 3), dtype=np.uint8)

session = requests.Session()
total = n_x * n_y
downloaded = 0
failed_tiles = []
sample_tile_headers = None
sample_tile_exif_keys = None
sample_tile_info_keys = None

for row, y in enumerate(range(y_min, y_max + 1)):
    for col, x in enumerate(range(x_min, x_max + 1)):
        url = TILE_URL.format(key=API_KEY, z=ZOOM, y=y, x=x)
        try:
            resp = session.get(url, timeout=10)
            resp.raise_for_status()
            content_type = resp.headers.get("Content-Type", "")
            if "image" not in content_type.lower():
                raise ValueError(f"non-image response ({content_type}): {resp.text[:160]}")
            pil_img = Image.open(BytesIO(resp.content))
            if sample_tile_headers is None:
                sample_tile_headers = {
                    k: v for k, v in resp.headers.items()
                    if k.lower() in {"date", "last-modified", "etag", "cache-control", "content-type"}
                }
                sample_tile_exif_keys = list(pil_img.getexif().keys())
                sample_tile_info_keys = sorted(pil_img.info.keys())
            img = np.array(pil_img.convert("RGB"))
            r0, r1 = row * TILE_SIZE, (row + 1) * TILE_SIZE
            c0, c1 = col * TILE_SIZE, (col + 1) * TILE_SIZE
            canvas[r0:r1, c0:c1] = img
        except Exception as e:
            print(f"  [WARN] tile ({x},{y}) failed: {e}")
            failed_tiles.append({"x": x, "y": y, "row": row, "col": col, "error": str(e)})
        downloaded += 1
        if downloaded % 10 == 0 or downloaded == total:
            print(f"  {downloaded}/{total} tiles done", end="\r")

successful_tiles = total - len(failed_tiles)
print(f"\nDone. Canvas: {canvas.shape}")
print(f"Successful tiles: {successful_tiles}/{total}")
if failed_tiles:
    print(f"Failed tiles: {len(failed_tiles)}")
if successful_tiles == 0:
    raise RuntimeError("All V-World tile downloads failed. Check API key, network, or TILE_URL.")

In [ ]:
west  = tile_to_lon(x_min, ZOOM)
east  = tile_to_lon(x_max + 1, ZOOM)
north = tile_to_lat(y_min, ZOOM)
south = tile_to_lat(y_max + 1, ZOOM)

transform = from_bounds(west, south, east, north, canvas.shape[1], canvas.shape[0])

with rasterio.open(
    OUTPUT_PATH, "w",
    driver="GTiff",
    height=canvas.shape[0],
    width=canvas.shape[1],
    count=3,
    dtype=np.uint8,
    crs=CRS.from_epsg(4326),
    transform=transform,
    compress="lzw",
) as dst:
    for i in range(3):
        dst.write(canvas[:, :, i], i + 1)

print(f"Saved: {OUTPUT_PATH}")
print(f"Bounds: W={west:.6f}, S={south:.6f}, E={east:.6f}, N={north:.6f}")

In [ ]:
def get_satellite_capabilities_metadata() -> dict:
    metadata = {"capabilities_url_template": CAPABILITIES_URL.replace("{key}", "{key}")}
    try:
        resp = session.get(CAPABILITIES_URL.format(key=API_KEY), timeout=20)
        resp.raise_for_status()
        root = ET.fromstring(resp.content)
        ns = {
            "wmts": "http://www.opengis.net/wmts/1.0",
            "ows": "http://www.opengis.net/ows/1.1",
        }
        service = root.find("ows:ServiceIdentification", ns)
        if service is not None:
            for key_name in ["Title", "Abstract", "ServiceType", "ServiceTypeVersion", "Fees", "AccessConstraints"]:
                el = service.find(f"ows:{key_name}", ns)
                metadata[f"service_{key_name.lower()}"] = (el.text or "").strip() if el is not None else None
        contents = root.find("wmts:Contents", ns)
        if contents is not None:
            for layer in contents.findall("wmts:Layer", ns):
                identifier = layer.find("ows:Identifier", ns)
                if identifier is None or identifier.text != "Satellite":
                    continue
                wgs84_bbox = layer.find("ows:WGS84BoundingBox", ns)
                lower = wgs84_bbox.find("ows:LowerCorner", ns).text.split() if wgs84_bbox is not None else None
                upper = wgs84_bbox.find("ows:UpperCorner", ns).text.split() if wgs84_bbox is not None else None
                resource = layer.find("wmts:ResourceURL", ns)
                resource_template = resource.attrib.get("template") if resource is not None else None
                if resource_template:
                    resource_template = resource_template.replace(API_KEY, "{key}")
                metadata["satellite_layer"] = {
                    "title": (layer.find("ows:Title", ns).text or "").strip(),
                    "identifier": "Satellite",
                    "format": (layer.find("wmts:Format", ns).text or "").strip(),
                    "tile_matrix_set": (layer.find("wmts:TileMatrixSetLink/wmts:TileMatrixSet", ns).text or "").strip(),
                    "wgs84_bbox": {
                        "west": float(lower[0]), "south": float(lower[1]),
                        "east": float(upper[0]), "north": float(upper[1]),
                    } if lower and upper else None,
                    "resource_url_template": resource_template,
                }
                break
    except Exception as e:
        metadata["capabilities_error"] = str(e)
    return metadata


metadata = {
    "area_name": AREA_NAME,
    "area_description": AREA_DESCRIPTION,
    "provider": "V-World",
    "service": "OGC WMTS",
    "layer": "Satellite",
    "zoom": ZOOM,
    "tile_size": TILE_SIZE,
    "requested_bbox_wgs84": BBOX,
    "actual_tile_bounds_wgs84": {"west": west, "south": south, "east": east, "north": north},
    "tile_range": {"x_min": x_min, "x_max": x_max, "y_min": y_min, "y_max": y_max},
    "tile_grid": {"n_x": n_x, "n_y": n_y, "total": total},
    "download": {
        "downloaded_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "successful_tiles": successful_tiles,
        "failed_tiles": len(failed_tiles),
        "failures_preview": failed_tiles[:50],
    },
    "output": {
        "path": str(OUTPUT_PATH),
        "metadata_path": str(METADATA_PATH),
        "width": int(canvas.shape[1]),
        "height": int(canvas.shape[0]),
        "bands": 3,
        "dtype": "uint8",
        "crs": "EPSG:4326",
    },
    "sample_tile_response": {
        "http_headers": sample_tile_headers,
        "jpeg_exif_keys": sample_tile_exif_keys,
        "pil_info_keys": sample_tile_info_keys,
    },
    "available_source_metadata": get_satellite_capabilities_metadata(),
    "acquisition_metadata": {
        "acquisition_date": None,
        "sensor": None,
        "note": "V-World WMTS Satellite tiles and WMTSCapabilities did not expose per-tile acquisition date or sensor metadata in this workflow. The sampled JPEG tile had no EXIF keys.",
    },
}

METADATA_PATH.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved metadata: {METADATA_PATH}")
print(json.dumps(metadata["acquisition_metadata"], indent=2, ensure_ascii=False))

In [ ]:
import matplotlib.pyplot as plt

with rasterio.open(OUTPUT_PATH) as src:
    preview = np.stack([src.read(1), src.read(2), src.read(3)], axis=-1)

fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(preview)
ax.set_title(f"V-World Aerial — {AREA_NAME} (zoom={ZOOM})")
ax.axis("off")
plt.tight_layout()
plt.show()